# AI-Driven Data Analysis with PandasAI (CBORG Edition)

This notebook demonstrates how to use **PandasAI 3.0+** to perform natural language analysis across multiple materialized views in the `ca-biositing` project. 

It is configured to use the **CBORG API** (Gemini models) at LBL.

### Step 1: Setup and Environment

We start by loading our environment variables and initializing the LLM. This notebook uses the `ai-exploration` Pixi environment which contains the modern `pandasai` library.

**Note for PandasAI 3.0:** We use a custom LLM class to connect to the CBORG gateway and a custom `ResponseParser` for interactive Plotly charts and DataFrame viewing.

In [ ]:
import sys
import os
from pathlib import Path

print(f"Python Executable: {sys.executable}")
print(f"Current Working Directory: {os.getcwd()}")

# Force discovery of local modules in analysis/ai_exploration
def find_setup_dir():
    # Try relative to root, then relative to current
    for p in [Path("analysis/ai_exploration"), Path(".")]:
        if (p / "sandbox_setup.py").exists():
            return p.resolve()
    return None

setup_dir = find_setup_dir()
if setup_dir:
    if str(setup_dir) not in sys.path:
        sys.path.insert(0, str(setup_dir))
        print(f"Added {setup_dir} to sys.path")
else:
    print("WARNING: Could not find sandbox_setup.py directory automatically.")

try:
    from sandbox_setup import init_sandbox, get_agent_with_metadata
    print("SUCCESS: sandbox_setup imported")
except ImportError as e:
    print(f"FAILURE: {e}")
    print(f"Path: {sys.path}")
    raise

llm, engine = init_sandbox(model_name="gemini-2.0-flash")

### Step 2: Database Connection

We connect to the database to fetch the materialized views. By default, this connects to the local development database.

In [ ]:
print("Database engine initialized via setup.py")

### Step 3: Load DataFrames

Fetch the materialized views into Pandas DataFrames for analysis.

In [ ]:
views = ["analysis_data_view", "usda_census_view", "analysis_average_view"]
print(f"Views targeted: {', '.join(views)}")

### Step 4: AI Analysis with Agent

Now we can query the data using natural language. PandasAI will generate and execute Python code to answer your questions.

**Interactive Charts:** We use a custom `ResponseParser` to ensure interactive Plotly visualizations and DataFrames are returned directly to the notebook.

In [ ]:
# Initialize the Agent with SQL Connectors and schema-aware metadata
agent = get_agent_with_metadata(llm, engine, views)

print(f"Agent initialized with {len(views)} views and custom SQL skill.")

### Step 5: Visualizations

You can also ask the AI to generate interactive charts.

In [ ]:
agent.chat("Show me an interactive plotly bar chart of the top 5 resources by volume from the analysis data")